In [ ]:
!pip install gTTS groq pandas streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
%%writefile app.py
import json
import re
import pandas as pd
import streamlit as st
from groq import Groq
from gtts import gTTS
import os

# ---------------------------------------------------------
# 1. PAGE SETUP & VIBRANT CUSTOM STYLING (CSS)
# ---------------------------------------------------------
st.set_page_config(
    page_title="🌈 Supercharged AI News Analyzer",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom Vibrant CSS Theme
st.markdown("""
<style>
    /* Main Background */
    .stApp {
        background: linear-gradient(135deg, #0f172a 0%, #1e1b4b 50%, #311042 100%);
        color: #f8fafc;
    }

    /* Sidebar Styling */
    section[data-testid="stSidebar"] {
        background-color: rgba(15, 23, 42, 0.85);
        border-right: 2px solid #8b5cf6;
    }

    /* Main Titles and Headers */
    h1 {
        background: linear-gradient(90deg, #ff007f, #7928ca, #00dfd8);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-weight: 800 !important;
    }

    h2, h3 {
        color: #38bdf8 !important;
    }

    /* Custom Cards / Metric Boxes */
    .metric-card {
        background: linear-gradient(135deg, rgba(255, 255, 255, 0.05), rgba(255, 255, 255, 0.1));
        border: 1px solid rgba(255, 255, 255, 0.2);
        backdrop-filter: blur(10px);
        border-radius: 12px;
        padding: 15px;
        text-align: center;
        box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    }

    /* Category Tags / Chips */
    .keyword-chip {
        display: inline-block;
        background: linear-gradient(90deg, #ec4899, #8b5cf6);
        color: white;
        padding: 5px 12px;
        margin: 3px;
        border-radius: 20px;
        font-size: 0.85rem;
        font-weight: bold;
    }

    /* Primary Buttons */
    .stButton>button {
        background: linear-gradient(90deg, #ff007f 0%, #7928ca 100%) !important;
        color: white !important;
        font-weight: bold !important;
        border: none !important;
        border-radius: 8px !important;
        padding: 10px 24px !important;
        transition: all 0.3s ease !important;
    }
    .stButton>button:hover {
        transform: scale(1.03);
        box-shadow: 0 0 15px rgba(236, 72, 153, 0.6);
    }
</style>
""", unsafe_allow_html=True)

# ---------------------------------------------------------
# 2. SIDEBAR CONFIGURATION
# ---------------------------------------------------------
st.sidebar.title("⚙️ Groq Control Panel")
api_key = st.sidebar.text_input("Enter Groq API Key:", type="password")

selected_model = st.sidebar.selectbox(
    "Select Model:",
    ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]
)

# Target Language Selection
target_language = st.sidebar.selectbox(
    "🌐 Summary Language:",
    ["English", "Spanish", "French", "German", "Hindi", "Chinese", "Arabic"]
)

# Language codes for Text-to-Speech (gTTS)
LANG_CODES = {
    "English": "en", "Spanish": "es", "French": "fr",
    "German": "de", "Hindi": "hi", "Chinese": "zh-CN", "Arabic": "ar"
}

client = None
if api_key.strip():
    try:
        client = Groq(api_key=api_key.strip())
    except Exception as e:
        st.sidebar.error(f"Error initializing Groq client: {e}")
else:
    st.sidebar.warning("⚠️ Enter your Groq API Key to enable analysis.")

# ---------------------------------------------------------
# 3. ENHANCED GROQ ANALYSIS FUNCTION
# ---------------------------------------------------------
def analyze_article_advanced(text, language):
    """Performs Category Prediction, Keyword Extraction, Sentiment, Fake News Check & Translation."""
    system_prompt = f"""
    You are an advanced NLP system specializing in news analysis, truth verification, and translation.
    Analyze the provided news article text and return ONLY a single valid JSON object.

    IMPORTANT: Provide the "summary" field translated into {language}.
    All other fields should remain in English.

    The JSON output MUST follow this schema strictly:
    {{
        "summary": "<Concise 2-3 sentence summary written in {language}>",
        "category": "<One of: Politics, Technology, Business, Health, Science, Entertainment, Sports, World News>",
        "category_confidence": <Integer 0-100>,
        "keywords": ["<keyword1>", "<keyword2>", "<keyword3>", "<keyword4>", "<keyword5>"],
        "sentiment": "<Positive, Negative, or Neutral>",
        "sentiment_score": <Integer 0-100>,
        "verdict": "<REAL, FAKE, or UNVERIFIED>",
        "fake_news_reasons": "<1-2 sentences explaining why it seems authentic or suspicious>"
    }}
    """

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Analyze this article:\n\n{text}"}
            ],
            model=selected_model,
            temperature=0.2,
            response_format={"type": "json_object"}
        )

        response_content = chat_completion.choices[0].message.content
        return json.loads(response_content)

    except Exception as e:
        return {"error": str(e)}

def generate_voice_summary(text, lang_code):
    """Converts text summary into audio using gTTS."""
    try:
        tts = gTTS(text=text, lang=lang_code, slow=False)
        audio_path = "summary_voice.mp3"
        tts.save(audio_path)
        return audio_path
    except Exception as e:
        return None

# ---------------------------------------------------------
# 4. MAIN APP UI DASHBOARD
# ---------------------------------------------------------
st.title("📰 AI News Intelligence & Truth Verification")
st.caption("Summarization • Multi-Language Translation • Voice Audio • Keyword Extraction • Truth Detection")

option = st.sidebar.radio(
    "Navigation:",
    ["Single Article Analyzer", "Upload & Search Dataset"]
)

# --- TAB 1: SINGLE ARTICLE ANALYZER ---
if option == "Single Article Analyzer":
    st.subheader("🔍 Real-Time Article Intelligence")
    input_text = st.text_area("Paste Article Content / News Text Here:", height=200)

    if st.button("🚀 Analyze Article", type="primary"):
        if not client:
            st.error("Please enter your Groq API Key in the sidebar.")
        elif not input_text.strip():
            st.warning("Please paste article content to proceed.")
        else:
            with st.spinner("Running AI models..."):
                result = analyze_article_advanced(input_text, target_language)

                if "error" in result:
                    st.error(f"API Error: {result['error']}")
                else:
                    st.success("Analysis Complete!")

                    # Metrics Row
                    col1, col2, col3 = st.columns(3)

                    with col1:
                        st.markdown(f"""
                        <div class="metric-card">
                            <small>CATEGORY PREDICTION</small>
                            <h3>🏷️ {result['category']}</h3>
                            <small>Confidence: {result['category_confidence']}%</small>
                        </div>
                        """, unsafe_allow_html=True)

                    with col2:
                        st.markdown(f"""
                        <div class="metric-card">
                            <small>SENTIMENT ANALYSIS</small>
                            <h3>🎭 {result['sentiment']}</h3>
                            <small>Strength: {result['sentiment_score']}%</small>
                        </div>
                        """, unsafe_allow_html=True)

                    with col3:
                        verdict = result["verdict"]
                        v_color = "#10b981" if verdict == "REAL" else ("#ef4444" if verdict == "FAKE" else "#f59e0b")
                        st.markdown(f"""
                        <div class="metric-card">
                            <small>TRUTH VERDICT</small>
                            <h3 style="color: {v_color} !important;">🛡️ {verdict}</h3>
                            <small>Fact Check Status</small>
                        </div>
                        """, unsafe_allow_html=True)

                    st.markdown("<br>", unsafe_allow_html=True)

                    # Summary & Key Features
                    st.markdown(f"### 📝 AI Summary ({target_language})")
                    st.info(result["summary"])

                    # Voice Summary (gTTS)
                    audio_file = generate_voice_summary(result["summary"], LANG_CODES.get(target_language, "en"))
                    if audio_file and os.path.exists(audio_file):
                        st.markdown("#### 🎧 Voice Summary")
                        st.audio(audio_file, format="audio/mp3")

                    # Download Summary Button
                    st.download_button(
                        label="📥 Download Summary (.txt)",
                        data=f"Summary ({target_language}):\n{result['summary']}\n\nCategory: {result['category']}\nVerdict: {result['verdict']}\nKeywords: {', '.join(result.get('keywords', []))}",
                        file_name="news_summary.txt",
                        mime="text/plain"
                    )

                    # Extracted Keywords
                    st.markdown("### 🔑 Extracted Keywords")
                    keywords_html = "".join([f'<span class="keyword-chip">{kw}</span>' for kw in result.get("keywords", [])])
                    st.markdown(keywords_html, unsafe_allow_html=True)

                    # Fake News Rationale
                    st.markdown("### 🛡️ Fact-Check Analysis")
                    st.write(result["fake_news_reasons"])

# --- TAB 2: DATASET UPLOAD & SEARCH ---
elif option == "Upload & Search Dataset":
    st.subheader("📂 News Dataset Search & Batch Processing")
    uploaded_file = st.file_uploader("Upload CSV Dataset", type=["csv"])

    if uploaded_file is not None:
        df = pd.read_csv(uploaded_file)
        st.write("### Dataset Preview")
        st.dataframe(df.head())

        text_col = st.selectbox("Select column containing Article Text:", df.columns)

        st.markdown("---")
        st.subheader("🔍 Keyword Search")
        search_query = st.text_input("Enter search keywords:")

        if search_query:
            filtered_df = df[df[text_col].astype(str).str.contains(search_query, case=False, na=False)]
            st.write(f"Found **{len(filtered_df)}** matching articles.")
            st.dataframe(filtered_df)

            if len(filtered_df) > 0 and st.button("Batch Analyze Top 3 Search Results"):
                if not client:
                    st.error("Please enter your Groq API Key in the sidebar.")
                else:
                    for idx, row in filtered_df.head(3).iterrows():
                        st.markdown(f"#### Article ID #{idx}")
                        article_content = str(row[text_col])

                        with st.spinner(f"Analyzing article #{idx}..."):
                            res = analyze_article_advanced(article_content, target_language)

                            if "error" in res:
                                st.error(f"Error: {res['error']}")
                            else:
                                st.write(f"**Summary ({target_language}):** {res['summary']}")
                                st.write(f"**Category:** {res['category']} | **Sentiment:** {res['sentiment']} | **Verdict:** {res['verdict']}")
                                st.write(f"**Keywords:** {', '.join(res.get('keywords', []))}")
                        st.markdown("---")

In [ ]:
import subprocess
import time

# Start Streamlit background process
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Pause brief moment for process init
time.sleep(3)

# Expose Streamlit via Cloudflare
!cloudflared tunnel --url http://localhost:8501

2026-08-14T07:43:02Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-14T07:43:02Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-14T07:43:05Z INF +--------------------------------------------------------------------------------------------+
2026-08-14T07:43:05Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-14T07:43:05Z INF |  https://allocated-consequently-spa-ross.trycloudflare